In [1]:
!python -V

Python 3.13.12


In [2]:
import binpan
binpan.__version__

'0.8.14'

# Create a Symbol Instance

The `Symbol` class is the main entry point in BinPan. It fetches candlestick (kline) data from the Binance API for a given trading pair and timeframe. The resulting DataFrame contains OHLCV data plus metadata like timestamps and taker volumes.

Key parameters:
- `symbol`: the trading pair (e.g. `"LTCBTC"`)
- `tick_interval`: candle timeframe (`"1m"`, `"5m"`, `"1h"`, `"1d"`, etc.)
- `time_zone`: display timezone for the index
- `limit`: number of candles to fetch

In [3]:
symbol = "dotusdc"
tick_interval = "1m"

In [4]:
sym = binpan.Symbol(symbol=symbol, tick_interval=tick_interval, time_zone="Europe/Madrid", limit = 500)
sym.df

2026-03-05	 20:31:05     INFO get_candles_by_time_stamps -> symbol=DOTUSDC tick_interval=1m start=2026-03-05 12:10:00 end=2026-03-05 20:30:59 limit=500
2026-03-05 20:31:05     INFO [panzer.binance.public.spot] Inicializando BinancePublicClient(market=spot)
2026-03-05 20:31:06     INFO [panzer.binance.public.spot] Rate limiter inicializado: max_per_minute=6000 safety_ratio=0.90


,Open time,Open,High,Low,Close,Volume,Close time,Quote volume,Trades,Taker buy base volume,Taker buy quote volume,Ignore,Open timestamp,Close timestamp
DOTUSDC 1m Europe/Madrid,,,,,,,,,,,,,,
2026-03-05 12:10:00+01:00,2026-03-05 12:10:00+01:00,1.538,1.538,1.538,1.538,0.00,2026-03-05 12:10:59.999000+01:00,0.00000,0,0.00,0.00000,0,1772709000000,1772709059999
2026-03-05 12:11:00+01:00,2026-03-05 12:11:00+01:00,1.540,1.540,1.539,1.540,821.87,2026-03-05 12:11:59.999000+01:00,1265.52416,11,508.31,782.79740,0,1772709060000,1772709119999
2026-03-05 12:12:00+01:00,2026-03-05 12:12:00+01:00,1.541,1.541,1.541,1.541,4.86,2026-03-05 12:12:59.999000+01:00,7.48926,1,4.86,7.48926,0,1772709120000,1772709179999
2026-03-05 12:13:00+01:00,2026-03-05 12:13:00+01:00,1.541,1.542,1.541,1.542,2187.08,2026-03-05 12:13:59.999000+01:00,3370.33605,5,2178.46,3357.05263,0,1772709180000,1772709239999
2026-03-05 12:14:00+01:00,2026-03-05 12:14:00+01:00,1.542,1.542,1.542,1.542,383.47,2026-03-05 12:14:59.999000+01:00,591.31074,3,0.00,0.00000,0,1772709240000,1772709299999
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-03-05 20:25:00+01:00,2026-03-05 20:25:00+01:00,1.515,1.515,1.515,1.515,269.47,2026-03-05 20:25:59.999000+01:00,408.24705,4,0.00,0.00000,0,1772738700000,1772738759999
2026-03-05 20:26:00+01:00,2026-03-05 20:26:00+01:00,1.515,1.515,1.514,1.515,331.11,2026-03-05 20:26:59.999000+01:00,501.61249,6,327.08,495.50704,0,1772738760000,1772738819999
2026-03-05 20:27:00+01:00,2026-03-05 20:27:00+01:00,1.517,1.517,1.515,1.515,2246.47,2026-03-05 20:27:59.999000+01:00,3405.38724,12,16.48,25.00016,0,1772738820000,1772738879999


# Add Technical Indicators

BinPan provides built-in technical indicators that are added as new columns to the Symbol's DataFrame. Here we add:
- **EMA (Exponential Moving Average)**: a trend-following indicator with a 200-period window.
- **VWAP (Volume-Weighted Average Price)**: cumulative over the data range, it shows the average price weighted by volume and is commonly used as a reference for fair value.

Indicators added with `inplace=True` (the default) are automatically included in subsequent `sym.plot()` calls.

In [5]:
sym.ema(200)
sym.vwap()

2026-03-05	 20:31:08     INFO New column: EMA_200
2026-03-05	 20:31:08     INFO New column: VWAP_D


DOTUSDC 1m Europe/Madrid
2026-03-05 12:10:00+01:00         NaN
2026-03-05 12:11:00+01:00    1.539667
2026-03-05 12:12:00+01:00    1.539675
2026-03-05 12:13:00+01:00    1.541120
2026-03-05 12:14:00+01:00    1.541219
                               ...   
2026-03-05 20:25:00+01:00    1.512076
2026-03-05 20:26:00+01:00    1.512078
2026-03-05 20:27:00+01:00    1.512093
2026-03-05 20:28:00+01:00    1.512093
2026-03-05 20:29:00+01:00    1.512096
Name: VWAP_D, Length: 500, dtype: float64

In [6]:
sym.plot()

'/home/nando/PycharmProjects/binpan_studio_dev/notebooks/last_plot.png'

# Atomic Trades Analysis

Atomic trades are the individual (non-aggregated) trades that occurred during the kline period. By fetching them, we can analyze the actual trade sizes and identify large buyers or sellers.

The `plot_atomic_trades_size()` method renders a bubble chart where bubble size represents trade quantity, allowing visual identification of significant market participants.

In [7]:
sym.get_atomic_trades()

2026-03-05	 20:31:11     INFO Obteniendo atomic trades de DOTUSDC por tiempo: 2026-03-05 11:10:00.000000 -> 2026-03-05 19:29:00.000000


Requesting atomic trades between 2026-03-05 12:10:00 and 2026-03-05 20:29:00


2026-03-05	 20:31:11     INFO Rango de IDs atómicos descubierto para DOTUSDC: 12616140 -> 12620491 (estimado 4352 trades)
2026-03-05	 20:31:12     INFO Peticiones API atomic trades DOTUSDC: 1 (ID actual: 12617140)
2026-03-05	 20:31:13     INFO Peticiones API atomic trades DOTUSDC: 2 (ID actual: 12618140)
2026-03-05	 20:31:13     INFO Peticiones API atomic trades DOTUSDC: 3 (ID actual: 12619140)
2026-03-05	 20:31:13     INFO Peticiones API atomic trades DOTUSDC: 4 (ID actual: 12620140)
2026-03-05	 20:31:14     INFO Peticiones API atomic trades DOTUSDC: 5 (ID actual: 12620499)
2026-03-05	 20:31:14     INFO Atomic trades DOTUSDC: 5 peticiones, 4359 trades obtenidos


,Trade Id,Price,Quantity,Quote quantity,Date,Timestamp,Buyer was maker,Best price match
DOTUSDC Europe/Madrid,,,,,,,,
0,12616140,1.540,3.44,5.29760,2026-03-05 12:11:03,1772709063816,False,True
1,12616141,1.540,300.33,462.50820,2026-03-05 12:11:03,1772709063816,False,True
2,12616142,1.540,115.37,177.66980,2026-03-05 12:11:03,1772709063817,False,True
3,12616143,1.540,46.97,72.33380,2026-03-05 12:11:03,1772709063888,False,True
4,12616144,1.539,146.81,225.94059,2026-03-05 12:11:05,1772709065656,True,True
...,...,...,...,...,...,...,...,...
4347,12620487,1.516,99.78,151.26648,2026-03-05 20:27:49,1772738869380,True,True
4348,12620488,1.516,1189.30,1802.97880,2026-03-05 20:27:49,1772738869380,True,True
4349,12620489,1.516,100.00,151.60000,2026-03-05 20:27:49,1772738869380,True,True


In [8]:
sym.plot_atomic_trades_size(logarithmic=True, max_size=40)

'/home/nando/PycharmProjects/binpan_studio_dev/notebooks/last_plot.png'

In the chart above, each trade is represented as a bubble whose size corresponds to the trade quantity. Color distinguishes taker buyers from taker sellers. The relationship between bubble size and price action reveals:
- **Exhaustion**: large taker buy volume without a corresponding price increase suggests buyer exhaustion.
- **Strength**: large taker buy volume followed by a price increase indicates strong buying pressure.
- **Hidden activity**: sometimes large taker buy entries are actually stop-loss executions or cancellations from the makers' side.

## Market Profile

The market profile aggregates volume at each price level, showing where the most trading activity occurred. This helps identify high-volume nodes (areas of price acceptance) and low-volume nodes (areas of price rejection).

In [9]:
sym.plot_market_profile(from_atomic_trades=True)

'/home/nando/PycharmProjects/binpan_studio_dev/notebooks/last_plot.png'

# Support and Resistance Levels

BinPan uses K-Means clustering on trade data to identify price levels where significant volume has accumulated. These levels often act as support or resistance zones.

The `support_resistance()` method returns lists of support and resistance price centroids (the center of each cluster, representing the most representative price for that zone). When `from_atomic=True`, it uses the granular atomic trade data for more precise level detection.

In [10]:
sym.support_resistance(from_atomic=True)

2026-03-05	 20:31:16  WARNING Clustering data...
2026-03-05	 20:31:16     INFO Updating support_resistance levels for DOTUSDC from atomic trades


([1.4881676624923856,
  1.5014972329801604,
  1.5164262849470558,
  1.5246811939589355,
  1.5387075236757188],
 [])

Once support and resistance levels are computed, they are automatically overlaid on the candlestick chart when calling `sym.plot()`. The horizontal lines mark the cluster centroids where the highest trade volume concentration was detected.

## Order Book Density

The order book snapshot shows the current distribution of limit orders (bids and asks) at different price levels, providing insight into near-term supply and demand zones.

In [ ]:
sym.plot(title="DOTUSDC S/R desde atomic trades (alta precisión)")

In [12]:
sym.get_orderbook()
sym.plot_orderbook_density()

'/home/nando/PycharmProjects/binpan_studio_dev/notebooks/last_plot.png'

## Order Book Depth (Accumulated)

The accumulated depth chart shows cumulative bid and ask volume at each price level, visualizing the total supply and demand around the current price.

By combining candlestick data, technical indicators, trade flow analysis (atomic trades), volume profiling (market profile), support/resistance levels, and order book density, you can build a more complete picture of the market microstructure to inform trading decisions.

## Maker / Taker Buy Ratios

The maker/taker buy ratio measures aggression: a ratio above 1 means taker buyers dominate (bullish pressure), while below 1 means taker sellers dominate (bearish pressure). A rolling window smooths the ratio over time.

In [13]:
sym.plot_orderbook()

'/home/nando/PycharmProjects/binpan_studio_dev/notebooks/last_plot.png'

---

For more examples, see the other notebooks in this repository: trading indicators, plotting, exchange operations, and database integration.

In [14]:
sym.plot_taker_maker_ratio_profile()

2026-03-05	 20:31:20     INFO Using klines data. For deeper info add trades data, example: my_symbol.get_agg_trades()


'/home/nando/PycharmProjects/binpan_studio_dev/notebooks/last_plot.png'

## Taker/Maker Ratio Profile

The ratio profile shows the distribution of maker/taker ratios across price levels, helping identify at which prices buying or selling aggression was strongest.

In [15]:
sym.get_maker_taker_buy_ratios(window=14)
sym.plot()

2026-03-05	 20:31:20     INFO Maker vs Taker buy ratio: 51 NaNs found. Converting to zeros.
2026-03-05	 20:31:20     INFO New column: Ratio_Taker/Maker_buy


'/home/nando/PycharmProjects/binpan_studio_dev/notebooks/last_plot.png'